In [ ]:
import cudf
import numpy as np
import cupy as cp
import pickle
import xgboost as xgb
import torch
from torch_geometric.nn import Node2Vec, SAGEConv
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import warnings
#warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# ÉTAPE 0 : Chargement, tri temporel, split, features de base
# ============================================================

# 1. Chargement GPU
df = cudf.read_csv('/kaggle/input/datasets/cisko6koman/data-tour-2026/test.csv')
df = df.sort_values('period')  # respecter la chronologie

# Split temporel : 80% des périodes pour l'entraînement
max_period = df['period'].max()
split_period = int(max_period * 0.8)
df_train = df[df['period'] <= split_period].copy()
df_test = df[df['period'] > split_period].copy()
print(f"Train : {len(df_train)} | Test : {len(df_test)}")
print(f"Fraude train : {df_train['fraud_flag'].mean()*100:.2f}%")
print(f"Fraude test  : {df_test['fraud_flag'].mean()*100:.2f}%")

# 2. Encodage des opérations (catégoriel sur GPU)
df_train['op_code'] = df_train['operation'].astype('category').cat.codes
df_test['op_code'] = df_test['operation'].astype('category').cat.codes

# 3. Construction du dictionnaire des comptes (uniquement sur train, + placeholder -1)
all_accounts_train = cudf.concat([
    df_train['origin_account'],
    df_train['destination_account']
]).unique().to_pandas()
# Ajouter un index pour les comptes inconnus (valeur -1)
account_to_id = {acc: idx for idx, acc in enumerate(all_accounts_train)}
UNK_ID = len(account_to_id)  # nouvel index pour les comptes jamais vus

# Sauvegarde pour usage futur
with open('account_to_id.pkl', 'wb') as f:
    pickle.dump(account_to_id, f)

# Fonction de mapping rapide (reste sur GPU avec merge)
def map_accounts(df, col_name, mapping, unk):
    # Convertir la colonne en pandas pour appliquer map, puis remettre sur GPU
    ids = df[col_name].to_pandas().map(mapping).fillna(unk).astype(np.int32)
    return cudf.Series(ids)

df_train['origin_id'] = map_accounts(df_train, 'origin_account', account_to_id, UNK_ID)
df_train['dest_id'] = map_accounts(df_train, 'destination_account', account_to_id, UNK_ID)
df_test['origin_id'] = map_accounts(df_test, 'origin_account', account_to_id, UNK_ID)
df_test['dest_id'] = map_accounts(df_test, 'destination_account', account_to_id, UNK_ID)

# 4. Features de base (uniquement avant transaction)
# On évite les soldes after et les erreurs pour rester réaliste.
features_base = [
    'period', 'op_code', 'amount',
    'origin_balance_before', 'destination_balance_before'
]
# Option : ajouter des ratios simples (pas de fuite)
df_train['amount_ratio_orig'] = df_train['amount'] / (df_train['origin_balance_before'].abs() + 1e-6)
df_train['amount_ratio_dest'] = df_train['amount'] / (df_train['destination_balance_before'].abs() + 1e-6)
df_test['amount_ratio_orig'] = df_test['amount'] / (df_test['origin_balance_before'].abs() + 1e-6)
df_test['amount_ratio_dest'] = df_test['amount'] / (df_test['destination_balance_before'].abs() + 1e-6)

features_base += ['amount_ratio_orig', 'amount_ratio_dest']

# Préparation des matrices pour XGBoost (converties en numpy uniquement pour XGB)
X_train_baseline = df_train[features_base].to_pandas()
y_train = df_train['fraud_flag'].to_pandas()
X_test_baseline = df_test[features_base].to_pandas()
y_test = df_test['fraud_flag'].to_pandas()

In [ ]:
# ============================================================
# TEST 1 : Degrés cuGraph + XGBoost
# ============================================================
import cugraph

# Graphe orienté uniquement sur l'entraînement
G_train = cugraph.Graph(directed=True)
G_train.from_cudf_edgelist(
    df_train, source='origin_id', destination='dest_id', edge_attr='amount'
)

# Calcul des degrés (in/out)
deg = G_train.degrees()  # colonnes : vertex, in_degree, out_degree

# Fusion avec train et test
def merge_degrees(df, deg_df):
    df = df.merge(deg_df, left_on='origin_id', right_on='vertex', how='left')
    df = df.rename(columns={'in_degree': 'o_in', 'out_degree': 'o_out'}).drop(columns=['vertex'])
    df = df.merge(deg_df, left_on='dest_id', right_on='vertex', how='left')
    df = df.rename(columns={'in_degree': 'd_in', 'out_degree': 'd_out'}).drop(columns=['vertex'])
    df = df.fillna(0)  # nouveaux comptes
    return df

df_train_deg = merge_degrees(df_train, deg)
df_test_deg = merge_degrees(df_test, deg)

features_deg = features_base + ['o_in', 'o_out', 'd_in', 'd_out']
X_train_deg = df_train_deg[features_deg].to_pandas()
X_test_deg = df_test_deg[features_deg].to_pandas()

# Poids de classe
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_xgb1 = xgb.XGBClassifier(
    n_estimators=300, max_depth=6,
    scale_pos_weight=scale_pos_weight,
    tree_method='hist', device='cuda',
    random_state=42
)
model_xgb1.fit(X_train_deg, y_train)
probs1 = model_xgb1.predict_proba(X_test_deg)[:, 1]
auc1 = roc_auc_score(y_test, probs1)
print(f"✅ Test 1 (Degrés) AUC = {auc1:.4f}")

In [ ]:
# ============================================================
# TEST 2 : Node2Vec + XGBoost
# ============================================================

# Récupérer le nombre total de nœuds (train + compte inconnu)
num_nodes = UNK_ID + 1  # car UNK_ID est le max index utilisé

# Construire edge_index pour PyG (seulement les arêtes d'entraînement)
src = df_train['origin_id'].to_numpy()
dst = df_train['dest_id'].to_numpy()
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long).cuda()

# Modèle Node2Vec
model_n2v = Node2Vec(
    edge_index, embedding_dim=32,
    walk_length=20, context_size=10,
    walks_per_node=10, sparse=True
).cuda()

loader = model_n2v.loader(batch_size=256, shuffle=True, num_workers=0)
optimizer = torch.optim.SparseAdam(model_n2v.parameters(), lr=0.01)

# Entraînement léger
model_n2v.train()
for epoch in range(3):
    total_loss = 0
    for pos_idx, neg_idx in loader:
        optimizer.zero_grad()
        loss = model_n2v.loss(pos_idx.cuda(), neg_idx.cuda())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Node2Vec epoch {epoch+1}, loss = {total_loss:.4f}")

# Récupérer les embeddings (CPU numpy)
embeddings = model_n2v().detach().cpu().numpy()  # shape (num_nodes, 32)

# Créer un DataFrame cuDF avec les embeddings
emb_cols = [f'emb_{i}' for i in range(32)]
df_emb = cudf.DataFrame(embeddings, columns=emb_cols)
df_emb['node_id'] = range(num_nodes)

# Fusionner pour origine et destination
def merge_emb(df, emb_df, prefix):
    df = df.merge(emb_df, left_on='origin_id', right_on='node_id', how='left', suffixes=('', f'_{prefix}'))
    df = df.drop(columns=['node_id'])
    rename_dict = {c: f'{prefix}_{c}' for c in emb_cols}
    df = df.rename(columns=rename_dict)
    return df

df_train_n2v = df_train_deg.copy()
df_test_n2v = df_test_deg.copy()

df_train_n2v = merge_emb(df_train_n2v, df_emb, 'orig')
df_train_n2v = merge_emb(df_train_n2v, df_emb.rename(columns={'node_id':'node_id_dest'}), 'dest')
df_test_n2v = merge_emb(df_test_n2v, df_emb, 'orig')
df_test_n2v = merge_emb(df_test_n2v, df_emb.rename(columns={'node_id':'node_id_dest'}), 'dest')

# Imputer les embeddings manquants (nouveaux comptes) par la moyenne
for col in df_train_n2v.columns:
    if 'emb_' in col:
        mean_val = df_train_n2v[col].mean()
        df_train_n2v[col] = df_train_n2v[col].fillna(mean_val)
        df_test_n2v[col] = df_test_n2v[col].fillna(mean_val)

features_n2v = features_deg + [c for c in df_train_n2v.columns if c.startswith('orig_emb_') or c.startswith('dest_emb_')]
X_train_n2v = df_train_n2v[features_n2v].to_pandas()
X_test_n2v = df_test_n2v[features_n2v].to_pandas()

model_xgb2 = xgb.XGBClassifier(
    n_estimators=300, max_depth=6,
    scale_pos_weight=scale_pos_weight,
    tree_method='hist', device='cuda',
    random_state=42
)
model_xgb2.fit(X_train_n2v, y_train)
probs2 = model_xgb2.predict_proba(X_test_n2v)[:, 1]
auc2 = roc_auc_score(y_test, probs2)
print(f"✅ Test 2 (Node2Vec) AUC = {auc2:.4f}")

In [ ]:
# ============================================================
# TEST 3 : GraphSAGE (inductif)
# ============================================================
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
import torch.optim as optim

# --- Construction des features de nœuds à partir de l'entraînement ---
# On agrège par compte (sur train uniquement) des statistiques simples.
# Pour chaque compte, on calcule :
# - nombre de transactions sortantes
# - nombre de transactions entrantes
# - montant moyen émis
# - montant moyen reçu
# - solde moyen avant émission
# - solde moyen avant réception

# Agrégation côté émetteur
orig_stats = df_train.groupby('origin_id').agg({
    'amount': ['count', 'mean'],
    'origin_balance_before': 'mean',
    'amount_ratio_orig': 'mean'
}).reset_index()
orig_stats.columns = ['node_id', 'out_deg', 'avg_out_amount', 'avg_orig_bal_before', 'avg_ratio_orig']

# Agrégation côté récepteur
dest_stats = df_train.groupby('dest_id').agg({
    'amount': ['count', 'mean'],
    'destination_balance_before': 'mean',
    'amount_ratio_dest': 'mean'
}).reset_index()
dest_stats.columns = ['node_id', 'in_deg', 'avg_in_amount', 'avg_dest_bal_before', 'avg_ratio_dest']

# Fusionner les deux profils pour chaque compte
node_feat = orig_stats.merge(dest_stats, on='node_id', how='outer').fillna(0)
# Ajouter le compte inconnu avec des zéros
unk_row = cudf.DataFrame({'node_id': [UNK_ID]})
for col in node_feat.columns:
    if col != 'node_id':
        unk_row[col] = 0.0
node_feat = cudf.concat([node_feat, unk_row], ignore_index=True)
node_feat = node_feat.sort_values('node_id').reset_index(drop=True)

# Convertir en tensor PyTorch (sur GPU)
x = torch.tensor(node_feat.drop(columns=['node_id']).to_pandas().values, dtype=torch.float).cuda()
# Vérifier que le nombre de lignes correspond à num_nodes
assert x.shape[0] == num_nodes

# Créer le graphe PyG à partir des arêtes d'entraînement
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long).cuda()
# Toutes les arêtes sont de la classe "transaction" (on peut mettre un edge_attr si utile)
data = Data(x=x, edge_index=edge_index)

# Définition d'un modèle GraphSAGE simple pour classification de lien
class GraphSAGELinkPredictor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(out_channels * 2, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1),
            torch.nn.Sigmoid()
        )

    def forward(self, x, edge_index, src, dst):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.2, training=self.training)
        h = self.conv2(h, edge_index)
        # Concaténer les embeddings source/destination pour la prédiction
        edge_feat = torch.cat([h[src], h[dst]], dim=1)
        return self.classifier(edge_feat).squeeze()

# Préparation des paires d'entraînement/test
src_train = torch.tensor(df_train['origin_id'].values, dtype=torch.long).cuda()
dst_train = torch.tensor(df_train['dest_id'].values, dtype=torch.long).cuda()
y_train_sage = torch.tensor(y_train.values, dtype=torch.float).cuda()

src_test = torch.tensor(df_test['origin_id'].values, dtype=torch.long).cuda()
dst_test = torch.tensor(df_test['dest_id'].values, dtype=torch.long).cuda()

# Entraînement rapide
device = torch.device('cuda')
model_sage = GraphSAGELinkPredictor(x.shape[1], 64, 32).to(device)
optimizer = optim.Adam(model_sage.parameters(), lr=0.01)
loss_fn = torch.nn.BCELoss()

# Masque pour gérer le déséquilibre via pondération
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]).cuda()

model_sage.train()
for epoch in range(10):
    optimizer.zero_grad()
    pred = model_sage(data.x, data.edge_index, src_train, dst_train)
    loss = F.binary_cross_entropy(pred, y_train_sage, pos_weight=pos_weight)
    loss.backward()
    optimizer.step()
    if epoch % 2 == 0:
        print(f"SAGE epoch {epoch}, loss = {loss.item():.4f}")

# Évaluation
model_sage.eval()
with torch.no_grad():
    probs_sage = model_sage(data.x, data.edge_index, src_test, dst_test).cpu().numpy()
auc3 = roc_auc_score(y_test, probs_sage)
print(f"✅ Test 3 (GraphSAGE) AUC = {auc3:.4f}")

In [ ]:
# ============================================================
# Résumé des performances
# ============================================================
print("\n🏁 RÉCAPITULATIF DES AUC")
print(f"Test 1 (Degrés + XGBoost)  : {auc1:.4f}")
print(f"Test 2 (Node2Vec + XGBoost) : {auc2:.4f}")
print(f"Test 3 (GraphSAGE pur)      : {auc3:.4f}")